# Demo: Deploying a Customized Model with SageMaker

This notebook walks through:
- Setting up environment
- Uploading dataset
- Fine-tuning a model with JumpStart
- Deploying and testing
- Cleaning up resources


## 1. Configuration
Edit the values in this cell to change the model, instance type, or dataset.

In [ ]:
# --- Tuneable config ---
MODEL_ID         = 'huggingface-text2text-flan-t5-small'
TRAINING_INSTANCE = 'ml.g4dn.xlarge'
INFERENCE_INSTANCE = 'ml.g4dn.xlarge'
LOCAL_DATASET    = 'dataset.jsonl'
S3_KEY_PREFIX    = 'fm-data'

# Hyperparameters
EPOCHS           = 2
LEARNING_RATE    = 3e-5
BATCH_SIZE       = 4

## 2. Setup

In [ ]:
# Install the correct AWS SageMaker Python SDK (v2.x)
!pip install 'sagemaker>=2.0,<3.0' --quiet

# Restart kernel so the newly installed package is available
import sys
if 'sagemaker' in sys.modules:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Import libraries
import boto3
import sagemaker
from sagemaker.session import Session

session = Session()
role    = sagemaker.get_execution_role()
bucket  = session.default_bucket()
region  = session.boto_region_name

print(f'Region : {region}')
print(f'Bucket : {bucket}')
print(f'Role   : {role}')

## 3. Upload Dataset

In [ ]:
# Upload dataset to S3
s3_path = session.upload_data(LOCAL_DATASET, bucket=bucket, key_prefix=S3_KEY_PREFIX)
print('Uploaded to:', s3_path)

## 4. Fine-Tune with JumpStart

In [ ]:
# Load JumpStart estimator
from sagemaker.jumpstart.estimator import JumpStartEstimator

estimator = JumpStartEstimator(
    model_id=MODEL_ID,
    role=role,
    instance_type=TRAINING_INSTANCE
)

In [ ]:
# Configure hyperparameters
estimator.set_hyperparameters(
    epoch=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE
)

In [ ]:
# Start training job (blocks until complete)
estimator.fit({'train': s3_path}, wait=True)

## 5. Deploy

In [ ]:
# Deploy the fine-tuned model to a real-time endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type=INFERENCE_INSTANCE
)
print('Endpoint name:', predictor.endpoint_name)

## 6. Test

In [ ]:
# Test the model
# Input must match the training data format: 'Customer (<tone>): <message>'
response = predictor.predict(
    {
        'inputs': 'Customer (angry): The product I received is damaged.',
        'parameters': {
            'max_new_tokens': 100,
            'do_sample': False
        }
    }
)

print(response[0]['generated_text'])

## 7. Cleanup
**Run this cell when you are done to avoid ongoing charges.**

In [ ]:
# Delete ALL resources to avoid ongoing charges
s3_client = boto3.client('s3', region_name=region)

# 1. Delete the endpoint (also removes endpoint config and model automatically)
try:
    predictor.delete_endpoint()
    print('✓ Endpoint deleted')
except Exception as e:
    print(f'Endpoint deletion skipped: {e}')

# 2. Delete uploaded dataset from S3
try:
    s3_client.delete_object(Bucket=bucket, Key=f'{S3_KEY_PREFIX}/{LOCAL_DATASET}')
    print('✓ S3 dataset deleted')
except Exception as e:
    print(f'S3 dataset deletion skipped: {e}')

# 3. Delete training job output artifacts from S3
try:
    training_job_name = estimator.latest_training_job.name
    prefix = f'{training_job_name}/output/'
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = response.get('Contents', [])
    if objects:
        s3_client.delete_objects(
            Bucket=bucket,
            Delete={'Objects': [{'Key': obj['Key']} for obj in objects]}
        )
        print(f'✓ Training artifacts deleted ({len(objects)} objects)')
    else:
        print('No training artifacts found in S3')
except Exception as e:
    print(f'Training artifact deletion skipped: {e}')

print('\nCleanup complete. Verify in the AWS Console that no endpoints or models remain.')